### P2.3 — Deep Learning Foundations | Python to Gen AI Course
### P2.3.10 — RNN and LSTM — Networks With Memory

---
## 🔁 Quick Recap — Where We Are

```
P2.3.8   ANN  →  Tabular data     (order does not matter)
P2.3.9   CNN  →  Image data       (spatial structure matters)
P2.3.10  RNN  →  Sequence data    (order and time matters)
```

ANN and CNN have one thing in common —  
they process each input **independently**, with no memory of what came before.

That works perfectly for house prices and images.  
But for sequences — text, audio, time series — **order is everything.**

```
"The movie was not bad at all"   →  Positive
"The movie was bad, not good"    →  Negative

Same words. Different order. Completely different meaning.
```

An ANN sees the same bag of words — it cannot tell these apart.  
An RNN reads the sequence in order — and remembers what came before.

---
## 🧠 The Core Idea — Memory

A regular (ANN) neuron:
```
Input  →  Neuron  →  Output
         (forgets immediately after)
```

An RNN neuron:
```
Input  →  Neuron  →  Output
            ↑  ↓
         Hidden State
         (memory passed to next step)
```

At every time step, the RNN neuron receives **two things:**
1. The current input (new word / new data point)
2. The hidden state from the **previous step** (memory of what came before)

It combines both, produces an output, and passes an updated hidden state forward.

```
Time Step 1:  Input('The')   + no memory    → Output + Hidden State h1
Time Step 2:  Input('cat')   + h1           → Output + Hidden State h2
Time Step 3:  Input('sat')   + h2           → Output + Hidden State h3
Time Step 4:  Input('on')    + h3           → Output + Hidden State h4
Time Step 5:  Input('the')   + h4           → Output + Hidden State h5
Time Step 6:  Input('mat')   + h5           → Final Output
```

By the final step, the hidden state carries information from every word.

<img src="rnn_unrolled.png" width="850"/>
<!-- Visual: RNN unrolled across 5 time steps. Each step shows: input word at top, RNN cell box in the middle, horizontal arrow passing hidden state (h) from left to right. Output arrow going up from each cell. Show h1, h2, h3... labels on the arrows between cells. Words: 'The', 'cat', 'sat', 'on', 'mat'. -->

---
## ⚠️ The Vanishing Gradient Problem

Plain RNNs have a serious limitation — **short memory.**

Recall from P2.3.4: gradients flow backwards through the network during backpropagation.  
In an RNN, gradients also flow backwards through **time steps.**

```
Step 100 ← Step 99 ← Step 98 ← ... ← Step 1

Each step, the gradient gets multiplied by a small number
After many steps: 0.9 × 0.9 × 0.9 × ... = almost zero
```

The gradient **vanishes** before it reaches early time steps.  
Early steps get almost no learning signal — the model forgets the distant past.

**Real consequence:**
```
Sentence: 'The cat, which was sitting on the old dusty mat in the corner, sat.'

Plain RNN remembers:  '...in the corner, sat'   (recent)
Plain RNN forgets:    'The cat...'              (distant — the subject!)

→ Cannot handle long-range dependencies
```

<img src="vanishing_gradient.png" width="750"/>
<!-- Visual: unrolled RNN with 10 steps. Gradient flow arrows going right to left (backprop direction). Arrows get progressively thinner and fade toward the left. Early steps show almost no gradient (near-zero). Label: 'Vanishing Gradient — early steps receive no learning signal'. -->

---
## 🔒 LSTM — The Fix for Short Memory

**Long Short-Term Memory (LSTM)** was designed specifically to solve the vanishing gradient problem.

An LSTM cell has a **cell state** — a separate memory track that runs through all steps.  
This cell state can carry information across long distances without the gradient disappearing.

The LSTM controls what to remember and what to forget using three **gates:**

```
───────────────────────────────────────────────────────────────
 Gate            What it decides
───────────────────────────────────────────────────────────────
 Forget Gate     What to erase from long-term memory
                 (e.g. forget the subject when a new sentence starts)

 Input Gate      What new information to add to long-term memory
                 (e.g. remember the new subject)

 Output Gate     What to pass forward as the hidden state
                 (e.g. what is relevant for the current prediction)
───────────────────────────────────────────────────────────────
```

These gates are learned during training — the LSTM figures out what is worth remembering.

> You do not need to implement LSTM gates yourself.  
> `keras.layers.LSTM()` and `torch.nn.LSTM()` do all of this internally.  
> Understanding the gates helps you understand **why LSTM works.**

<img src="lstm_gates.png" width="800"/>
<!-- Visual: LSTM cell diagram. Two horizontal tracks: TOP track = Cell State (long-term memory, flows straight across). BOTTOM track = Hidden State (short-term, passed step to step). Three gate boxes intersecting the cell state track: Forget Gate (trash bin icon), Input Gate (plus icon), Output Gate (filter icon). Labels on each gate explaining what it does. -->

---
## 🆚 RNN vs LSTM — When to Use Which

```
────────────────────────────────────────────────────────────────
                    Plain RNN              LSTM
────────────────────────────────────────────────────────────────
 Memory             Short-term only        Short + Long term
 Long sequences     Struggles              Handles well
 Speed              Faster                 Slightly slower
 Parameters         Fewer                  More
 Use today          Rarely used alone      Standard choice
 Best for           Very short sequences   Most sequence tasks
────────────────────────────────────────────────────────────────
```

> In practice: **always start with LSTM.**  
> Plain RNN is mostly used for teaching — LSTM is what gets used in production.

---
## 💻 Let's Build It — LSTM for Sentiment Classification

Task: Classify a movie review as Positive or Negative.  
Dataset: IMDB — 50,000 movie reviews, pre-labeled.

```
Input  → sequence of words (up to 200 words)
Output → 1 neuron, sigmoid → Positive (1) or Negative (0)
```

We use PyTorch here — so you see the full training loop explicitly.

In [ ]:
# ── KERAS VERSION — clean and fast ────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers

# ── DATA ──────────────────────────────────────────────────────────
# IMDB built into Keras — words already converted to integers
# num_words=10000 → keep only the 10,000 most common words
max_words  = 10000
max_length = 200

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(
    num_words=max_words
)

print(f"Train reviews : {len(X_train)}")
print(f"Test  reviews : {len(X_test)}")
print(f"Sample label  : {y_train[0]}  (1=Positive, 0=Negative)")
print(f"Sample review : {X_train[0][:10]}...  (word indices)")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 1 : SPLIT + PREPARE
# IMDB is already split — we pad sequences to equal length
# ══════════════════════════════════════════════════════════════════

# Reviews have different lengths — pad/trim all to max_length
# Short reviews get zeros added at the start (pre-padding)
# Long reviews get trimmed to max_length
X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=max_length)
X_test  = keras.preprocessing.sequence.pad_sequences(X_test,  maxlen=max_length)

print(f"Shape after padding: {X_train.shape}")
print(f"Each review is now exactly {max_length} word indices long")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 2 : TRAIN
# ══════════════════════════════════════════════════════════════════

model = keras.Sequential([

    # Embedding — converts word indices to dense vectors
    # Each word gets a 64-dimensional vector that is learned during training
    # This is what captures meaning — similar words get similar vectors
    layers.Embedding(max_words, 64, input_length=max_length),

    # LSTM — reads the sequence, maintains memory across all 200 words
    # return_sequences=False — we only need the output of the final step
    layers.LSTM(64),

    # Dropout — prevent overfitting
    layers.Dropout(0.3),

    # Output — binary classification (positive / negative)
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ── TRAIN ─────────────────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=5,
    batch_size=128,
    verbose=1
)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 3 : EVALUATE
# ══════════════════════════════════════════════════════════════════
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss,  test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

train_acc = round(train_acc * 100, 2)
test_acc  = round(test_acc  * 100, 2)

print("=== PHASE 3 : EVALUATE ===")
print(f"  Train Accuracy : {train_acc}%")
print(f"  Test  Accuracy : {test_acc}%")

gap = train_acc - test_acc
if gap < 5:
    print("  ✅ Good Fit — train and test scores are close")
elif gap > 15:
    print("  ⚠️  Overfitting — add more Dropout or reduce LSTM size")
else:
    print("  ✅ Acceptable — small gap, generalises reasonably well")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PHASE 4 : INFERENCE
# ══════════════════════════════════════════════════════════════════

# Get the word index map from IMDB
word_index = keras.datasets.imdb.get_word_index()

def predict_sentiment(review_text):
    """
    Convert a raw text review to a sentiment prediction.
    Maps each word to its IMDB index, pads to max_length, runs inference.
    """
    import re
    words  = re.sub(r'[^a-z ]', '', review_text.lower()).split()
    indices = [word_index.get(w, 2) for w in words]  # 2 = unknown word
    indices = indices[:max_length]   # trim if too long

    import numpy as np
    padded = keras.preprocessing.sequence.pad_sequences([indices], maxlen=max_length)
    prob   = model.predict(padded, verbose=0)[0][0]

    label  = "POSITIVE" if prob > 0.5 else "NEGATIVE"
    conf   = round(prob * 100 if prob > 0.5 else (1 - prob) * 100, 1)
    return label, conf

print("=== PHASE 4 : INFERENCE ===")
reviews = [
    "This movie was absolutely fantastic. The acting was brilliant.",
    "Terrible film. Complete waste of time. Boring and predictable.",
    "Not bad. Some good moments but overall quite average.",
]
for review in reviews:
    label, conf = predict_sentiment(review)
    print(f"  '{review[:50]}...'")
    print(f"   → {label}  ({conf}% confident)\n")

---
## 📌 What Is the Embedding Layer?

You noticed the network starts with `layers.Embedding()`.  
This is a critical piece that converts words into numbers the LSTM can understand.

```
Word Index  →  Embedding Layer  →  Dense Vector

'cat'  (index 42)   →   [0.23, -0.14, 0.89, ..., 0.31]  (64 numbers)
'dog'  (index 87)   →   [0.21, -0.11, 0.91, ..., 0.29]  (64 numbers)
'car'  (index 156)  →   [-0.41, 0.73, -0.12, ..., 0.05] (64 numbers)
```

Words with similar meaning get similar vectors.  
`cat` and `dog` end up close together in the 64-dimensional space.  
`car` ends up far from both.

The embedding weights are **learned during training** —  
the network figures out which meanings matter for the task.

> This is the bridge between text and numbers that every NLP model needs.  
> In modern systems (BERT, GPT), this idea scales to billions of dimensions.

<img src="embedding_layer.png" width="750"/>
<!-- Visual: LEFT column — words ('cat', 'dog', 'car', 'run'). Middle — Embedding lookup table (grid of numbers). RIGHT — each word maps to a row of numbers (the vector). Below: 2D scatter plot showing 'cat' and 'dog' close together, 'car' far away. Label: 'Similar words → similar vectors'. -->

---
## 🌍 Where RNN / LSTM Is Used in the Real World

```
────────────────────────────────────────────────────────────────
 Application               What RNN/LSTM Does
────────────────────────────────────────────────────────────────
 Gmail Smart Compose        Predicts next words as you type
 Google Translate (older)   Translates sentence word by word
 Siri / Alexa               Converts your speech to text
 Stock Price Prediction      Predicts next value from historical prices
 Music Generation            Predicts next note from previous notes
 Subtitle Generation         Generates captions from audio sequence
 Anomaly Detection           Detects abnormal patterns in time series
────────────────────────────────────────────────────────────────
```

> Note: Modern language tasks (translation, text generation) have largely  
> moved from LSTM to Transformers — a more powerful architecture.  
> But LSTM remains important for time series, audio, and shorter sequences.

<img src="rnn_real_world.png" width="750"/>
<!-- Visual: 3×2 icon grid. Gmail / Google Translate / Siri / Stock chart / Music notes / Subtitles. Each tile with one-line label. Dark background. -->

---
## ✅ Summary — What You Learned

| Concept | Key Point |
|---|---|
| Why RNN exists | ANN has no memory — order is invisible. RNN reads sequences in order |
| Hidden State | Memory passed from step to step — carries context forward |
| Vanishing Gradient | Gradients shrink to near-zero over long sequences — plain RNN forgets |
| LSTM | Three gates — Forget, Input, Output — controls what to remember long-term |
| Cell State | LSTM's long-term memory track — carries information across many steps |
| RNN vs LSTM | Always prefer LSTM — plain RNN rarely used in practice |
| Embedding Layer | Converts words to dense vectors — similar words get similar vectors |
| IMDB Dataset | 50,000 movie reviews — binary sentiment classification benchmark |
| Real world | Speech recognition, text autocomplete, stock prediction, translation |

---

## 🎉 Module Complete — P2.3 Deep Learning Foundations

```
P2.3.1   Why Deep Learning
P2.3.2   The Neuron
P2.3.3   From Neuron to Network
P2.3.4   How a Network Learns
P2.3.5   Overfitting and How to Fix It
P2.3.6   TensorFlow, Keras and PyTorch
P2.3.7   How to Break Down a DL Problem
P2.3.8   ANN — Tabular Data
P2.3.9   CNN — Images
P2.3.10  RNN / LSTM — Sequences
```

You now have a solid foundation in every major Deep Learning concept —  
from a single neuron all the way to sequence-aware memory networks.

**The next module begins the Gen AI journey. 🚀**